## 音声のスパース性に基づくビームフォーマ

In [1]:
import wave 
import pyroomacoustics as pa
import numpy as np
import scipy.signal as sp
import scipy

from calc_steering_vector import calculate_steering_vector

In [23]:
# 時間周波数マスクを推定する
def estimate_mask(input_vectors, steering_vectors, omega):
    """
    input_vectors: マイクロホン入力信号 (num_microhones, freq_bins, time_frames)
    steering_vectors: ステアリングベクトル (freq_bins, target_signal_range, time_frames)
    omega: 目的音の範囲 (num_sources, target_signal_range=72)
    """
    inner_product = np.einsum("kim,mkt->kit", np.conjugate(steering_vectors), input_vectors)
    """inner_product: (freq_bins, target_signal_range, time_frames)"""
    n_omega = np.shape(omega)[1]
    estimate_doas = np.argmax(np.abs(inner_product), axis=1)
    """estimate_doas: (freq_bins, time_frames)"""
    estimate_doas_mask = np.identity(n_omega)[estimate_doas]
    """estimate_doas_mask: (freq_bins, time_frames, target_signal_range)"""
    mask = np.einsum("kti,si->skt", estimate_doas_mask, omega)
    """mask: (num_sources, freq_bins, time_frames)"""
    return mask

In [4]:
# マスクと入力信号から共分散行列を推定
def estimate_covariance_matrix(x, mask):
    """
    x: 入力信号 (num_microphones, freq_bins, time_frames)
    mask: 音源ごとの時間周波数マスク (num_sources, freq_bins, time_frames)
    """
    # 目的音の共分散行列を推定する
    Rs = np.einsum("skt,mkt,nkt->skmn", mask, x, np.conjugate(x))
    """Rs: (num_sources, freq_bins, num_microphones, num_microphones)"""
    sum_target_mask = np.sum(mask, axis=2)
    """sum_target_mask: (num_sources, freq_bins)"""
    Rs = Rs / np.maximum(sum_target_mask, 1.e-18)[..., None, None]
    """Rs: (num_sources, freq_bins, num_microphones, num_microphones)"""
    # 雑音の共分散行列を推定する
    Rn = np.einsum("skt,mkt,nkt->skmn", 1-mask, x, np.conjugate(x))
    """Rn: (num_sources, freq_bins, num_microphones, num_microphones)"""
    sum_noise_mask = np.sum(1-mask, axis=2)
    """sum_noise_mask: (num_sources, freq_bins)"""
    Rn = Rn / np.maximum(sum_noise_mask, 1.e-18)[..., None, None]
    # 固有値分解をして半正定値行列に変換
    w, v = np.linalg.eigh(Rs)
    """w: (num_sources, freq_bins, num_microphones), v: (num_sources, freq_bins, num_microphones, num_microphones)"""
    Rs_org = Rs.copy()
    w[np.real(w) < 1.e-18] = 1.e-18 # 固有値が0より小さい場合は0に置き換える
    Rs = np.einsum("skmi,ski,skni->skmn", v, w, np.conjugate(v))
    """Rn: (num_sources, freq_bins, num_microphones, num_microphones)"""
    w, v = np.linalg.eigh(Rn)
    """w: (num_sources, freq_bins, ), v: (num_sources, freq_bins, num_microphones, )"""
    Rn_org = Rn.copy()
    w[np.real(w) < 1.e-18] = 1.e-18 # 固有値が0より小さい場合は0に置き換える
    Rn = np.einsum("skmi,ski,skni->skmn", v, w, np.conjugate(v))
    """Rn: (num_sources, freq_bins, num_microphones, num_microphones)"""
    return Rs, Rn

In [2]:
# 音源のスパース性を仮定し、共分散行列からステアリングベクトルを推定する
def estimate_steering_vector(Rs):
    """
    Rs: 共分散行列 (num_sources, freq_bins, num_microphones, num_microphones)
    """
    # 固有値分解を実施して最大固有値を与える固有ベクトルを取得
    w, v = np.linalg.eigh(Rs)
    """w: (num_sources, freq_bins, num_microphones), v: (num_sources, freq_bins, num_microphones, num_microphones)"""
    steering_vector = v[..., -1]
    """steering_vector: (num_sources, freq_bins, num_microphones)"""
    return steering_vector

In [5]:
# スパース性に基づく分離
def execute_sparse(x, mask):
    """
    x: (num_microphones, freq_bins, time_frames)
    mask: (num_sources, freq_bins, time_frames)
    """
    c_hat = np.einsum("skt,mkt->mskt", mask, x)
    return c_hat

In [6]:
# 遅延和アレイ
def execute_dsbf(x, steering_vector):
    """
    x: (num_microphones, freq_bins, time_frames)
    steering_vector: (num_sources, freq_bins, num_microphones)
    """
    s_hat = np.einsum("skm,mkt->skt", np.conjugate(steering_vector), x)
    """s_hat: (num_sources, freq_bins, time_frames)"""
    # ステアリングベクトルを掛ける
    c_hat = np.einsum("skt,skm->mskt", s_hat, steering_vector)
    """c_hat: (num_microphones, num_sources, freq_bins, time_frames)"""
    return c_hat

In [7]:
# MVDR
def execute_mvdr(x, Rn, steering_vector):
    """
    x: (num_microphones, freq_bins, time_frames)
    Rn: (num_sources, freq_bins, num_microphones, num_microphones)
    steering_vector: (num_sources, freq_bins, num_microphones)
    """
    # 共分散行列の逆行列を計算する
    Rn_inverse = np.linalg.pinv(Rn)
    """Rn_inverse: (num_sources, freq_bins, num_microphones, num_microphones)"""
    # 分離フィルタを計算する
    Rn_inverse_a = np.einsum("skmn,skn->skm", Rn_inverse, steering_vector) # 分子
    """Rn_inverse_a: (num_sources, freq_bins, num_microphones)"""
    a_H_Rn_inverse_a = np.einsum("skn,skn->sk", np.conjugate(steering_vector), Rn_inverse_a) # 分母
    """a_H_Rn_inverse_a: (num_sources, freq_bins)"""
    w_mvdr = Rn_inverse_a / np.maximum(a_H_Rn_inverse_a, 1.e-18)[:, :, None]
    """w_mvdr: (num_sources, freq_bins, num_microphones)"""
    # 分離フィルタを掛ける
    s_hat = np.einsum("skm,mkt->skt", np.conjugate(w_mvdr), x)
    """s_hat: (num_sources, freq_bins, time_frames)"""
    # ステアリングベクトルを掛ける（マイクロホン入力信号中の目的音成分を推定）
    c_hat = np.einsum("skt,skm->mskt", s_hat, steering_vector)
    """c_hat: (num_microphones, num_sources, freq_bins, time_frames)"""
    return c_hat

In [8]:
# MVDR2を実行（共分散行列のみから計算）
def execute_mvdr2(x, Rs, Rn):
    """
    x: (num_microphones, freq_bins, time_frames)
    Rs: (num_sources, freq_bins, num_microphones, num_microphones)
    Rn: (num_sources, freq_bins, num_microphones, num_microphones)
    """
    # 共分散行列の逆行列を計算する
    Rn_inverse = np.linalg.pinv(Rn)
    """Rn_inverse: (num_sources, freq_bins, num_microphones, num_microphones)"""
    # フィルタを計算する
    Rn_inverse_Rs = np.einsum("skmi,skin->skmn", Rn_inverse, Rs)
    """Rn_inverse_Rs: (num_sources, freq_bins, num_microphones, num_microphones)"""
    w_mvdr = Rn_inverse_Rs / np.maximum(np.trace(Rn_inverse_Rs, axis1=-2, axis2=-1), 1.e-18)[..., None, None]
    """w_mvdr: (num_sources, freq_bins, num_microphones, num_microphones)"""
    # フィルタを掛ける
    c_hat = np.einsum("skmn,mkt->nskt", np.conjugate(w_mvdr), x)
    """c_hat: (num_microphones, num_sources, freq_bins, time_frames)"""
    return c_hat

In [9]:
# MaxSNR
def execute_max_snr(x, y):
    """
    x: (num_microphones, freq_bins, time_frames)
    y: (num_microphones, freq_bins, time_frames)
    """
    # 雑音の共分散行列
    Rn = np.average(np.einsum("mkt,nkt->ktmn", y, np.conjugate(y)), axis=1)
    """Rn: (freq_bins, num_microphones, num_microphones)"""
    # 入力共分散行列
    Rs = np.average(np.einsum("mkt,nkt->ktmn", x, np.conjugate(x)), axis=1)
    """Rs: (freq_bins, num_microphones, num_microphones)"""
    # 周波数の数を取得
    Nk = np.shape(Rs)[0]
    # 一般化固有値分解
    max_snr_filter = None
    for k in range(int(Nk)):
        w, v = scipy.linalg.eigh(Rs[k, ...], Rn[k, ...])
        """w: (num_microphones, ), v: (num_microphones, num_microphones)"""
        if max_snr_filter is None:
            max_snr_filter = v[None, :, -1]
        else:
            max_snr_filter = np.concatenate((max_snr_filter, v[None, :, -1]), axis=0)
    """max_snr_filter: (freq_bins, num_microphones)"""
    Rs_w = np.einsum("kmn,kn->km", Rs, max_snr_filter)
    """Rs_w: (freq_bins, num_microphones)"""
    beta = Rs_w[:, 0] / np.einsum("km,km->k", np.conjugate(max_snr_filter), Rs_w)[:, None]
    """beta: (freq_bins, )"""
    w_max_snr = beta[:, None, :] * max_snr_filter[..., None]
    """w_max_snr: (freq_bins, num_microphones, num_microphones)"""
    # フィルタを掛ける
    c_hat = np.einsum("kim,ikt->mkt", np.conjugate(w_max_snr), x)
    """c_hat: (num_microphones, freq_bins, time_frames)"""
    return c_hat

In [10]:
# MaxSNR
def execute_max_snr2(x, Rs, Rn):
    """
    x: (num_microphones, freq_bins, time_frames)
    Rs: (num_sources, freq_bins, num_microphones, num_microphones)
    Rn: (num_sources, freq_bins, num_microphones, num_microphones)
    """
    # 音源数を取得
    Ns = np.shape(Rs)[0]
    # 周波数の数を取得
    Nk = np.shape(Rs)[1]
    # 一般化固有値分解
    max_snr_filter = None
    max_snr_filter_all = None
    for s in range(int(Ns)):
        for k in range(int(Nk)):
            w, v = scipy.linalg.eigh(Rs[s, k, ...], Rn[s, k, ...])
            """w: (num_microphones, ), v: (num_microphones, num_microphones)"""
            if k == 0:
                max_snr_filter = v[None, :, -1]
            else:
                max_snr_filter = np.concatenate((max_snr_filter, v[None, :, -1]), axis=0)
        if s == 0:
            max_snr_filter_all = max_snr_filter[None, ...]
        else:
            max_snr_filter_all = np.concatenate((max_snr_filter_all, max_snr_filter[None, ...]), axis=0)
    """max_snr_filter_all: (num_sources, freq_bins, num_microphones)"""
    Rs_w = np.einsum("skmn,skn->skm", Rs, max_snr_filter_all)
    """Rs_w: (num_sources, freq_bins, num_microphones)"""
    beta = Rs_w / np.einsum("skm,skm->sk", np.conjugate(max_snr_filter_all), Rs_w)[:, :, None]
    """beta: (num_sources, freq_bins, num_microphones)"""
    w_max_snr = beta[:, :, None, :] * max_snr_filter_all[:, :, :, None]
    """w_max_snr: (num_sources, freq_bins, num_microphones, num_microphones)"""
    # フィルタを掛ける
    c_hat = np.einsum("skim,ikt->mskt", np.conjugate(w_max_snr), x)
    """c_hat: (num_microphones, num_sources, freq_bins, time_frames)"""
    return c_hat

In [11]:
# MWFを実行
def execute_mwf(x, Rs, Rn):
    """
    x: (num_microphones, freq_bins, time_frames)
    Rs: (num_sources, freq_bins, num_microphones, num_microphones)
    Rn: (num_sources, freq_bins, num_microphones, num_microphones)
    """
    # 入力信号に対する共分散行列の逆行列を計算
    Rx_inverse = np.linalg.pinv(Rs + Rn)
    """Rx_inverse: (num_sources, freq_bins, num_microphones, num_microphones)"""
    # フィルタ生成
    W_mwf = np.einsum("skmi,skin->skmn", Rx_inverse, Rs)
    """W_mwf: (num_sources, freq_bins, num_microphones, num_microphones)"""
    # フィルタを掛ける
    c_hat = np.einsum("skim,ikt->mskt", np.conjugate(W_mwf), x)
    """c_hat: (num_microphones, num_sources, freq_bins, time_frames)"""
    return c_hat

In [12]:
# SNRを測る
def calculate_snr(target, out):
    """
    target: 目的音 (num_samples, )
    out: 雑音除去後の信号 (num_samples, )
    """
    wave_length = np.minimum(np.shape(target)[0], np.shape(out)[0])
    # 消し残った雑音
    target = target[:wave_length]
    out = out[:wave_length]
    noise = target - out
    snr = 10. * np.log10(np.sum(np.square(target)) / np.sum(np.square(noise)))
    return snr

In [13]:
def modify_angle_diff(diff):
    diff = np.where(diff < -np.pi, diff + np.pi * 2, diff)
    diff = np.where(diff > np.pi, diff - np.pi * 2, diff)
    return diff

In [24]:
if __name__ == "__main__":
    # 乱数の種を初期化
    np.random.seed(0)
    # 畳み込みに用いる波形
    clean_wave_files = ["./CMU_ARCTIC/cmu_us_aew/wav/arctic_a0001.wav", "./CMU_ARCTIC/cmu_us_axb/wav/arctic_a0002.wav"]
    # 雑音だけの区間のフレーム数
    n_noise_only = 40000
    # 音源数
    n_sources = len(clean_wave_files)
    # 音声波形の長さを調べる
    n_samples = 0
    # ファイルを読み込む
    for clean_wave_file in clean_wave_files:
        wav = wave.open(clean_wave_file)
        if n_samples<wav.getnframes():
            n_samples=wav.getnframes()
        wav.close()
    clean_data = np.zeros([n_sources, n_samples])

    # ファイルを読み込む
    s = 0
    for clean_wave_file in clean_wave_files:
        wav = wave.open(clean_wave_file)
        data = wav.readframes(wav.getnframes())
        data = np.frombuffer(data, dtype=np.int16)
        data = data/np.iinfo(np.int16).max
        clean_data[s, :wav.getnframes()] = data
        wav.close()
        s = s+1

    # シミュレーションのパラメータ
    n_sim_sources = 2
    # サンプリングレート [Hz]
    sample_rate = 16000
    # フレームサイズ
    N = 1024
    # 周波数の数
    Nk = N / 2 + 1
    # 各ビンの周波数
    freqs = np.arange(0, Nk, 1) * sample_rate / N
    # 音声と雑音の比率 [dB]
    SNR = 90.
    # 方位角の閾値
    azimuth_thresh = 30
    # 部屋の大きさ
    room_dim = np.r_[10.0, 10.0, 10.0]
    # マイクロホンアレイを置く部屋の場所
    mic_array_loc = room_dim / 2 + np.random.randn(3) * 0.1
    # マイクロホンアレイのマイクロホン配置
    mic_directions = np.array(
        [[np.pi/2, theta/180 * np.pi] for theta in np.arange(0, 361, 180)]
    )
    distance = 0.02
    mic_alignments = np.zeros((3, mic_directions.shape[0]), dtype=mic_directions.dtype)
    mic_alignments[0, :] = np.cos(mic_directions[:, 1]) * np.sin(mic_directions[:, 0]) 
    mic_alignments[1, :] = np.sin(mic_directions[:, 1]) * np.sin(mic_directions[:, 0]) 
    mic_alignments[2, :] = np.cos(mic_directions[:, 0])
    mic_alignments *= distance

    # マイクロホン数
    n_channels = np.shape(mic_alignments)[0]
    # get the microphone array
    R  = mic_alignments + mic_array_loc[:, None]
    """R: (3D-coordinate(x,y,z)=3, num_microphones)"""
    # 部屋を生成する
    room = pa.ShoeBox(room_dim, fs=sample_rate, max_order=0)
    # room = pa.ShoeBox(room_dim, fs=sample_rate, max_order=17, absorption=0.4) # 残響がある場合
    room_no_noise_left = pa.ShoeBox(room_dim, fs=sample_rate, max_order=0)
    room_no_noise_right = pa.ShoeBox(room_dim, fs=sample_rate, max_order=0)
    # 用いるマイクロホンアレイの情報を設置する
    room.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    room_no_noise_left.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    room_no_noise_right.add_microphone_array(pa.MicrophoneArray(R, fs=room.fs))
    # 音源の場所
    doas  = np.array(
        [[np.pi/2, np.pi],
        [np.pi/2, 0]]
        )
    """doas: (num_sources, source_direction)"""
    # 音源の誤差
    doas_error = np.array(
        [[0, 5 / 180 * np.pi], 
        [0, 5 / 180 * np.pi]]
    )
    # 音源とマイクロホンの距離
    distance = 1.
    doas2 = doas + doas_error
    source_locations = np.zeros((3, doas2.shape[0]), dtype=doas2.dtype)
    """source_locations: (xyz, num_sources)"""
    source_locations[0,  :] = np.cos(doas2[:, 1]) * np.sin(doas2[:, 0]) 
    source_locations[1,  :] = np.sin(doas2[:, 1]) * np.sin(doas2[:, 0])
    source_locations[2,  :] = np.cos(doas2[:, 0])
    source_locations *= distance
    source_locations += mic_array_loc[:, None] # マイクロホンアレイからの相対位置→絶対位置

    # ステアリングベクトルを算出するための仮想的な音源方向
    virtual_doas = np.array(
        [[np.pi/2, theta/180 * np.pi] for theta in np.arange(0, 360, 5)]
    )
    virtual_source_locations = np.zeros((3, virtual_doas.shape[0]), dtype=virtual_doas.dtype)
    """virtual_source_locations: (xyz, num_sources)"""
    virtual_source_locations[0,  :] = np.cos(virtual_doas[:, 1]) * np.sin(virtual_doas[:, 0]) 
    virtual_source_locations[1,  :] = np.sin(virtual_doas[:, 1]) * np.sin(virtual_doas[:, 0])
    virtual_source_locations[2,  :] = np.cos(virtual_doas[:, 0])
    virtual_source_locations *= 100
    virtual_source_locations += mic_array_loc[:, None] # マイクロホンアレイからの相対位置→絶対位置
    # 仮想的な音源方向（0°, 5°,・・・, 355°）のステアリングベクトル作成
    virtual_steering_vectors = calculate_steering_vector(R, virtual_source_locations, freqs, is_use_far=True)
    """virtual_steering_vectors: (freq_bins, num_virtual_sources=72, num_microphones)"""

    # 所望音の方向から±thresh度以内
    omega = np.array([np.abs(modify_angle_diff(virtual_doas[:, 1] - doas[s, 1])) < azimuth_thresh / 180 * np.pi for s in range(n_sim_sources)]).astype(np.float)
    """omega: (n_sources, num_virtual_sources=72)"""

    # 各音源をシミュレーションに追加する
    for s in range(n_sim_sources):
        clean_data[s] /= np.std(clean_data[s])
        room.add_source(source_locations[:, s], signal=clean_data[s])
        if s == 0:
            room_no_noise_left.add_source(source_locations[:, s], signal=clean_data[s])
        if s == 1:
            room_no_noise_right.add_source(source_locations[:, s], signal=clean_data[s])

    # シミュレーションを回す
    room.simulate(snr=SNR)
    room_no_noise_left.simulate(snr=90)
    room_no_noise_right.simulate(snr=90)

    # 畳み込んだ波形を取得する
    multi_conv_data = room.mic_array.signals
    """multi_conv_data: (num_channels, num_samples)"""
    multi_conv_data_left_no_noise = room_no_noise_left.mic_array.signals
    """multi_conv_data_left_no_noise: (num_channels, num_samples)"""
    multi_conv_data_right_no_noise = room_no_noise_right.mic_array.signals
    """multi_conv_data_right_no_noise: (num_channels, num_samples)"""

    # 短時間フーリエ変換
    f, t, stft_data = sp.stft(multi_conv_data, fs=sample_rate, window="hann", nperseg=N)
    """f: (freq_bins,), t: (1,), stft_data:(num_microphones, freq_bins, time_frames)"""

    # # DOA情報を使って分離
    # y_doa = execute_doa_sparse_separation(stft_data, virtual_steering_vectors, omega)

    # 時間周波数マスクを推定
    tf_mask = estimate_mask(stft_data, virtual_steering_vectors, omega)

    # 共分散行列とステアリングベクトルを推定
    Rs, Rn = estimate_covariance_matrix(stft_data, tf_mask)
    desired_steering_vectors = estimate_steering_vector(Rs)

    # 各フィルタを実行する
    sparse_out = execute_sparse(stft_data, tf_mask)
    dsbf_out = execute_dsbf(stft_data, desired_steering_vectors)
    mvdr_out = execute_mvdr(stft_data, Rn, desired_steering_vectors)
    mvdr2_out = execute_mvdr2(stft_data, Rs, Rn)
    max_snr2_out = execute_max_snr2(stft_data, Rs, Rn)
    mwf_out = execute_mwf(stft_data, Rs, Rn)

    # 評価するマイクロホン
    eval_mic_index = 0

    # 時間領域の波形に戻す
    t, sparse_out_left = sp.istft(sparse_out[eval_mic_index, 0], fs=sample_rate, window="hann", nperseg=N)
    t, sparse_out_right = sp.istft(sparse_out[eval_mic_index, 1], fs=sample_rate, window="hann", nperseg=N)
    t, dsbf_out_left = sp.istft(dsbf_out[eval_mic_index, 0], fs=sample_rate, window="hann", nperseg=N)
    t, dsbf_out_right = sp.istft(dsbf_out[eval_mic_index, 1], fs=sample_rate, window="hann", nperseg=N)
    t, mvdr_out_left = sp.istft(mvdr_out[eval_mic_index, 0], fs=sample_rate, window="hann", nperseg=N)
    t, mvdr_out_right = sp.istft(mvdr_out[eval_mic_index, 1], fs=sample_rate, window="hann", nperseg=N)
    t, mvdr2_out_left = sp.istft(mvdr2_out[eval_mic_index, 0], fs=sample_rate, window="hann", nperseg=N)
    t, mvdr2_out_right = sp.istft(mvdr2_out[eval_mic_index, 1], fs=sample_rate, window="hann", nperseg=N)
    t, max_snr2_out_left = sp.istft(max_snr2_out[eval_mic_index, 0], fs=sample_rate, window="hann", nperseg=N)
    t, max_snr2_out_right = sp.istft(max_snr2_out[eval_mic_index, 1], fs=sample_rate, window="hann", nperseg=N)
    t, mwf_out_left = sp.istft(mwf_out[eval_mic_index, 0], fs=sample_rate, window="hann", nperseg=N)
    t, mwf_out_right = sp.istft(mwf_out[eval_mic_index, 1], fs=sample_rate, window="hann", nperseg=N)

    # SNRを測る
    snr_pre = calculate_snr(multi_conv_data_left_no_noise[0, ...], multi_conv_data[0, ...]) + calculate_snr(multi_conv_data_right_no_noise[0, ...], multi_conv_data[0, ...])
    snr_sparse_post = calculate_snr(multi_conv_data_left_no_noise[0, ...], sparse_out_left) + calculate_snr(multi_conv_data_right_no_noise[0, ...], sparse_out_right)
    snr_dsbf_post = calculate_snr(multi_conv_data_left_no_noise[0, ...], dsbf_out_left) + calculate_snr(multi_conv_data_right_no_noise[0, ...], dsbf_out_right)
    snr_mvdr_post = calculate_snr(multi_conv_data_left_no_noise[0, ...], mvdr_out_left) + calculate_snr(multi_conv_data_right_no_noise[0, ...], mvdr_out_right)
    snr_mvdr2_post = calculate_snr(multi_conv_data_left_no_noise[0, ...], mvdr2_out_left) + calculate_snr(multi_conv_data_right_no_noise[0, ...], mvdr2_out_right)
    snr_max_snr2_post = calculate_snr(multi_conv_data_left_no_noise[0, ...], max_snr2_out_left) + calculate_snr(multi_conv_data_right_no_noise[0, ...], max_snr2_out_right)
    snr_mwf_post = calculate_snr(multi_conv_data_left_no_noise[0, ...], mwf_out_left) + calculate_snr(multi_conv_data_right_no_noise[0, ...], mwf_out_right)
    snr_pre /= 2
    snr_sparse_post /= 2
    snr_dsbf_post /= 2
    snr_mvdr_post /= 2
    snr_mvdr2_post /= 2
    snr_max_snr2_post /= 2
    snr_mwf_post /= 2
    print("result ΔSNR [dB]")
    print("sparse:{:.2f}".format(snr_sparse_post-snr_pre))
    print("dsbf:{:.2f}".format(snr_dsbf_post-snr_pre))
    print("mvdr:{:.2f}".format(snr_mvdr_post-snr_pre))
    print("mvdr2:{:.2f}".format(snr_mvdr2_post-snr_pre))
    print("max_snr2:{:.2f}".format(snr_max_snr2_post-snr_pre))
    print("mwf:{:.2f}".format(snr_mwf_post-snr_pre))


[3.14159265 3.05432619 2.96705973 2.87979327 2.7925268  2.70526034
 2.61799388 2.53072742 2.44346095 2.35619449 2.26892803 2.18166156
 2.0943951  2.00712864 1.91986218 1.83259571 1.74532925 1.65806279
 1.57079633 1.48352986 1.3962634  1.30899694 1.22173048 1.13446401
 1.04719755 0.95993109 0.87266463 0.78539816 0.6981317  0.61086524
 0.52359878 0.43633231 0.34906585 0.26179939 0.17453293 0.08726646
 0.         0.08726646 0.17453293 0.26179939 0.34906585 0.43633231
 0.52359878 0.61086524 0.6981317  0.78539816 0.87266463 0.95993109
 1.04719755 1.13446401 1.22173048 1.30899694 1.3962634  1.48352986
 1.57079633 1.65806279 1.74532925 1.83259571 1.91986218 2.00712864
 2.0943951  2.18166156 2.26892803 2.35619449 2.44346095 2.53072742
 2.61799388 2.70526034 2.7925268  2.87979327 2.96705973 3.05432619]
result ΔSNR [dB]
sparse:7.27
dsbf:0.77
mvdr:17.19
mvdr2:12.39
max_snr2:14.28
mwf:12.94


In [ ]:
result ΔSNR [dB]
sparse:6.39
dsbf:0.75
mvdr:11.78
mvdr2:10.33
max_snr2:10.82
mwf:10.88

result ΔSNR [dB]
sparse:7.27
dsbf:0.77
mvdr:17.19
mvdr2:12.39
max_snr2:14.28
mwf:12.94
